# Melanoma & Nevus Dermoscopy Dataset - Improved Multimodal (EfficientNet-B0 + Focal Loss + TTA)

Áp dụng 3 cải tiến ưu tiên cao so với `06_train_multimodal.ipynb`:
1. **Backbone EfficientNet-B0** thay ResNet18 (feature 1280 chiều thay vì 512)
2. **Focal Loss** thay `BCEWithLogitsLoss` + `pos_weight` — nhắm trực tiếp vào điểm yếu đã phát hiện: sensitivity gần như không cải thiện ở bản trước
3. **Test-Time Augmentation (TTA)** — trung bình dự đoán trên 4 góc nhìn (gốc, flip ngang, flip dọc, flip cả 2) khi đánh giá test, không cần train lại

##### 1. Setup — chạy lại image pipeline

In [1]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = os.path.exists('/kaggle/input')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    pipeline_path = '/content/drive/MyDrive/Melanoma/04_image_pipeline.ipynb'
elif IN_KAGGLE:
    pipeline_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if '04_image_pipeline.ipynb' in files:
            pipeline_path = os.path.join(root, '04_image_pipeline.ipynb')
            break
    if pipeline_path is None:
        raise FileNotFoundError(
            'Không tìm thấy 04_image_pipeline.ipynb trong /kaggle/input - '
            'nhớ upload file này vào dataset'
        )
else:
    pipeline_path = '04_image_pipeline.ipynb'

print('pipeline_path:', pipeline_path)
%run {pipeline_path}

Mounted at /content/drive
pipeline_path: /content/drive/MyDrive/Melanoma/04_image_pipeline.ipynb
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Giải nén /content/drive/MyDrive/Melanoma/images.zip ra /content/images ...
Xong.
IN_COLAB: True
IMAGES_DIR: /content/images
PROCESSED_DIR: /content/drive/MyDrive/Melanoma/data/processed
Train: (12696, 25) | Val: (2752, 25) | Test: (2685, 25)
Số file không tồn tại (trong 20 mẫu đầu): 0
Số batch - train: 397 | val: 86 | test: 84
images shape : torch.Size([32, 3, 224, 224])
labels shape : torch.Size([32]) | vài label đầu: [0, 0, 1, 1, 1, 1, 0, 1]
metadata shape: torch.Size([32, 13])


In [2]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


##### 2. Model — EfficientNet-B0 + metadata MLP

Giống kiến trúc `MultimodalModel` ở `06_train_multimodal.ipynb`, chỉ đổi backbone: `efficientnet_b0`, bỏ `classifier` cuối (thay `nn.Identity()`) để lấy feature **1280 chiều** (thay vì 512 của ResNet18).

In [3]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


class MultimodalModel(nn.Module):
    def __init__(self, n_metadata_features, image_feature_dim=1280, metadata_hidden_dim=32):
        super().__init__()

        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        backbone.classifier = nn.Identity()  # bỏ classifier cuối, lấy thẳng feature 1280 chiều
        self.image_branch = backbone

        self.metadata_branch = nn.Sequential(
            nn.Linear(n_metadata_features, metadata_hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Linear(image_feature_dim + metadata_hidden_dim, 1)

    def forward(self, image, metadata):
        image_features = self.image_branch(image)
        metadata_features = self.metadata_branch(metadata)
        combined = torch.cat([image_features, metadata_features], dim=1)
        return self.classifier(combined)


model = MultimodalModel(n_metadata_features=len(METADATA_COLS)).to(device)
print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 179MB/s]


Linear(in_features=1312, out_features=1, bias=True)


##### 3. Sanity check — forward pass thử 1 batch

In [4]:
model.eval()
with torch.no_grad():
    sample_images, sample_labels, sample_metadata = next(iter(train_loader))
    sample_output = model(sample_images.to(device), sample_metadata.to(device))

print('Output shape:', sample_output.shape)  # kỳ vọng (batch, 1)

Output shape: torch.Size([32, 1])


##### 4. Loss — Focal Loss (thay BCEWithLogitsLoss + pos_weight)

`alpha` tính tương tự logic `pos_weight` trước đó (tỷ lệ nevus trong train) để upweight lớp melanoma (thiểu số). `gamma=2.0` là giá trị chuẩn, tập trung hơn vào các mẫu khó phân loại thay vì chỉ cân bằng tần suất lớp như BCE có trọng số thông thường.

In [5]:
import torch.optim as optim
from torchvision.ops import sigmoid_focal_loss

n_nevus = (df_train['diagnosis'] == 'nevus').sum()
n_melanoma = (df_train['diagnosis'] == 'melanoma').sum()

FOCAL_ALPHA = n_nevus / (n_nevus + n_melanoma)
FOCAL_GAMMA = 2.0
print(f'FOCAL_ALPHA: {FOCAL_ALPHA:.4f} | FOCAL_GAMMA: {FOCAL_GAMMA}')


def criterion(outputs, targets):
    return sigmoid_focal_loss(outputs, targets, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, reduction='mean')


optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

FOCAL_ALPHA: 0.6011 | FOCAL_GAMMA: 2.0


##### 5. Hàm train 1 epoch / đánh giá (không TTA — dùng cho theo dõi val mỗi epoch)

In [6]:
from sklearn.metrics import roc_auc_score, recall_score, accuracy_score, confusion_matrix


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, labels, metadata in loader:
        images = images.to(device)
        metadata = metadata.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images, metadata)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    for images, labels, metadata in loader:
        images = images.to(device)
        metadata = metadata.to(device)
        labels_t = labels.float().unsqueeze(1).to(device)

        outputs = model(images, metadata)
        loss = criterion(outputs, labels_t)
        total_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

    preds = [1 if p >= threshold else 0 for p in all_probs]

    metrics = {
        'loss': total_loss / len(loader.dataset),
        'accuracy': accuracy_score(all_labels, preds),
        'sensitivity': recall_score(all_labels, preds),
        'auc': roc_auc_score(all_labels, all_probs),
    }
    return metrics, all_labels, preds, all_probs

##### 6. Training loop với early stopping (có resume từ checkpoint)

Nếu `CHECKPOINT_PATH` đã tồn tại (ví dụ do lần chạy trước bị ngắt giữa chừng), tự động load lại model + optimizer + epoch đã dừng + best_val_auc, rồi train tiếp từ đó thay vì bắt đầu lại từ epoch 1. Checkpoint giờ lưu dạng dict (model, optimizer, epoch, best_val_auc) thay vì chỉ lưu trọng số model như trước — vẫn đọc được checkpoint định dạng cũ (chỉ có trọng số) nếu có, nhưng khi đó phải train lại từ epoch 1 vì không có thông tin epoch/best_auc đã lưu.

In [7]:
EPOCHS = 15
PATIENCE = 3

if IN_COLAB:
    CHECKPOINT_PATH = os.path.join(DRIVE_ROOT, 'best_model_improved.pt')
elif IN_KAGGLE:
    CHECKPOINT_PATH = '/kaggle/working/best_model_improved.pt'
else:
    CHECKPOINT_PATH = 'best_model_improved.pt'

print('CHECKPOINT_PATH:', CHECKPOINT_PATH)

start_epoch = 1
best_val_auc = 0.0
epochs_no_improve = 0

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        # Định dạng mới - có đủ thông tin để resume đúng epoch/best_val_auc
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_auc = checkpoint['best_val_auc']
        print(f'Tìm thấy checkpoint - resume từ epoch {start_epoch}, best_val_auc={best_val_auc:.4f}')
    else:
        # Định dạng cũ - chỉ có trọng số model, không biết epoch/best_val_auc đã train tới đâu
        model.load_state_dict(checkpoint)
        print('Tìm thấy checkpoint định dạng cũ (chỉ có trọng số) - đã load trọng số, '
              'nhưng train lại từ epoch 1 vì không biết best_val_auc/epoch trước đó')
else:
    print('Không có checkpoint cũ - train từ đầu')

for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics, _, _, _ = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_metrics['auc'])

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} "
          f"| val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
          f"val_sensitivity={val_metrics['sensitivity']:.4f} val_auc={val_metrics['auc']:.4f}")

    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        epochs_no_improve = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'best_val_auc': best_val_auc,
        }, CHECKPOINT_PATH)
        print(f'  -> Lưu checkpoint mới (val_auc={best_val_auc:.4f})')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'Early stopping tại epoch {epoch} (không cải thiện sau {PATIENCE} epoch)')
            break

print('\nBest val AUC:', best_val_auc)

CHECKPOINT_PATH: /content/drive/MyDrive/Melanoma/best_model_improved.pt
Không có checkpoint cũ - train từ đầu
Epoch 01 | train_loss=0.0640 | val_loss=0.0562 val_acc=0.7980 val_sensitivity=0.7227 val_auc=0.8717
  -> Lưu checkpoint mới (val_auc=0.8717)
Epoch 02 | train_loss=0.0534 | val_loss=0.0553 val_acc=0.8249 val_sensitivity=0.7070 val_auc=0.8846
  -> Lưu checkpoint mới (val_auc=0.8846)
Epoch 03 | train_loss=0.0475 | val_loss=0.0530 val_acc=0.8289 val_sensitivity=0.7588 val_auc=0.8952
  -> Lưu checkpoint mới (val_auc=0.8952)
Epoch 04 | train_loss=0.0426 | val_loss=0.0559 val_acc=0.8270 val_sensitivity=0.7468 val_auc=0.8944
Epoch 05 | train_loss=0.0388 | val_loss=0.0553 val_acc=0.8216 val_sensitivity=0.7902 val_auc=0.8974
  -> Lưu checkpoint mới (val_auc=0.8974)
Epoch 06 | train_loss=0.0349 | val_loss=0.0540 val_acc=0.8310 val_sensitivity=0.7893 val_auc=0.8993
  -> Lưu checkpoint mới (val_auc=0.8993)
Epoch 07 | train_loss=0.0322 | val_loss=0.0570 val_acc=0.8289 val_sensitivity=0.7782 

##### 7. Hàm đánh giá có TTA (Test-Time Augmentation)

Với mỗi ảnh, dự đoán trên 4 góc nhìn (gốc, flip ngang, flip dọc, flip cả 2 — đều là các biến đổi đã dùng lúc train nên hợp lệ), lấy trung bình xác suất. Loss vẫn tính trên view gốc để so sánh công bằng với cách đánh giá không-TTA.

In [8]:
@torch.no_grad()
def evaluate_tta(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    for images, labels, metadata in loader:
        images = images.to(device)
        metadata = metadata.to(device)
        labels_t = labels.float().unsqueeze(1).to(device)

        views = [
            images,
            torch.flip(images, dims=[3]),  # flip ngang
            torch.flip(images, dims=[2]),  # flip dọc
            torch.flip(images, dims=[2, 3]),  # flip cả ngang và dọc
        ]

        outputs_per_view = [model(v, metadata) for v in views]

        loss = criterion(outputs_per_view[0], labels_t)
        total_loss += loss.item() * images.size(0)

        probs_per_view = [torch.sigmoid(o) for o in outputs_per_view]
        probs_avg = torch.stack(probs_per_view).mean(dim=0)

        all_probs.extend(probs_avg.cpu().numpy().flatten())
        all_labels.extend(labels.numpy())

    preds = [1 if p >= threshold else 0 for p in all_probs]

    metrics = {
        'loss': total_loss / len(loader.dataset),
        'accuracy': accuracy_score(all_labels, preds),
        'sensitivity': recall_score(all_labels, preds),
        'auc': roc_auc_score(all_labels, all_probs),
    }
    return metrics, all_labels, preds, all_probs

##### 8. Đánh giá cuối cùng trên test set với TTA (chỉ chạy 1 lần, threshold=0.5)

In [10]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

test_metrics, test_labels, test_preds, test_probs = evaluate_tta(model, test_loader, criterion, device)

print('=== Test set metrics (TTA, threshold=0.5) ===')
for k, v in test_metrics.items():
    print(f'{k}: {v:.4f}')

print('\nConfusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):')
print(confusion_matrix(test_labels, test_preds))

=== Test set metrics (TTA, threshold=0.5) ===
loss: 0.0763
accuracy: 0.8313
sensitivity: 0.8180
auc: 0.9011

Confusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):
[[1369  261]
 [ 192  863]]


##### 9. Tune lại threshold trên val set (Youden's J, dùng TTA cho nhất quán)

In [11]:
import numpy as np

val_metrics, val_labels, _, val_probs = evaluate_tta(model, val_loader, criterion, device)

thresholds = np.arange(0.01, 1.0, 0.01)
best_threshold = 0.5
best_j = -1.0

for t in thresholds:
    preds = [1 if p >= t else 0 for p in val_probs]
    tn, fp, fn, tp = confusion_matrix(val_labels, preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    j = sensitivity + specificity - 1

    if j > best_j:
        best_j = j
        best_threshold = t

print(f"Threshold mặc định: 0.50")
print(f"Threshold tối ưu (Youden's J trên val, TTA): {best_threshold:.2f} (J={best_j:.4f})")

Threshold mặc định: 0.50
Threshold tối ưu (Youden's J trên val, TTA): 0.50 (J=0.6694)
